In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

NUM_CUSTOMERS = 1000000      # how many customers to generate
NUM_ORDERS    = 5000000      # how many orders to generate (total)

countries   = ["USA", "Canada", "Germany", "France", "UK", "Spain", "Italy", "Sweden"]
departments = ["Sales", "Engineering", "HR", "Finance", "Marketing", "Support"]
cities      = ["New York", "San Francisco", "Toronto", "Berlin", "Paris", "London", "Madrid", "Rome", "Stockholm"]
products    = ["Laptop", "Phone", "Headphones", "Monitor", "Keyboard", "Mouse", "Tablet", "Printer"]

countries_col   = F.array(*[F.lit(c) for c in countries])
departments_col = F.array(*[F.lit(d) for d in departments])
cities_col      = F.array(*[F.lit(c) for c in cities])
products_col    = F.array(*[F.lit(p) for p in products])

StatementMeta(, 57b9a557-a0cb-4ae9-9d05-8f914b662759, 3, Finished, Available, Finished, False)

In [2]:

customers_df = (
    spark.range(NUM_CUSTOMERS)  
        .withColumn("id", (F.col("id") + 1).cast("bigint"))
        .withColumn("name", F.concat(F.lit("Customer_"), F.col("id")))
        .withColumn(
            "country",
            F.element_at(
                countries_col,
                (F.floor(F.rand() * F.lit(len(countries))) + 1).cast("int")
            )
        )
        .withColumn(
            "city",
            F.element_at(
                cities_col,
                (F.floor(F.rand() * F.lit(len(cities))) + 1).cast("int")
            )
        )
        .withColumn(
            "email",
            F.concat(
                F.lower(F.regexp_replace("name", " ", "")),
                F.lit("@example.com")
            )
        )
        .withColumn(
            "phone",
            F.format_string(
                "+1%010d",
                (F.floor(F.rand() * F.lit(10_000_000_000))).cast("bigint")
            )
        )
        .select("id", "name", "country", "email", "city", "phone")
)

customers_df.write.mode('overwrite').saveAsTable("customers")



StatementMeta(, 57b9a557-a0cb-4ae9-9d05-8f914b662759, 4, Finished, Available, Finished, False)

In [6]:
orders_df = (
    spark.range(NUM_ORDERS)
        .withColumn("order_id", (F.col("id") + 1).cast("bigint"))
        .withColumn(
            "customer_id",
            (F.floor(F.rand() * F.lit(NUM_CUSTOMERS)) + 1).cast("bigint")
        )
        .withColumn(
            "product",
            F.element_at(
                products_col,
                (F.floor(F.rand() * F.lit(len(products))) + 1).cast("int")
            )
        )
        .withColumn(
            "price",
            F.round(F.rand() * F.lit(495) + F.lit(5), 2)
        )
        .withColumn(
            "quantity",
            (F.floor(F.rand() * F.lit(10)) + 1).cast("int")
        )
        .withColumn(
            "total",
            F.col("price") * F.col("quantity")
        )
        .withColumn(
            "order_date",
            F.expr("date_sub(current_date(), cast(rand() * 365 as int))")
        )
        .select("order_id", "customer_id", "product", "price", "quantity", "total", "order_date")
)

orders_df.write.mode('append').saveAsTable("orders")


StatementMeta(, 57b9a557-a0cb-4ae9-9d05-8f914b662759, 8, Finished, Available, Finished, False)

In [ ]:
orders_df = (
    spark.range(NUM_ORDERS)
        .withColumn("order_id", (F.col("id") + 1).cast("bigint"))
        .withColumn(
            "customer_id",
            (F.floor(F.rand() * F.lit(NUM_CUSTOMERS)) + 1).cast("bigint")
        )
        .withColumn(
            "product",
            F.element_at(
                products_col,
                (F.floor(F.rand() * F.lit(len(products))) + 1).cast("int")
            )
        )
        .withColumn(
            "price",
            F.round(F.rand() * F.lit(495) + F.lit(5), 2)
        )
        .withColumn(
            "quantity",
            (F.floor(F.rand() * F.lit(10)) + 1).cast("int")
        )
        .withColumn(
            "total",
            F.col("price") * F.col("quantity")
        )
        .withColumn(
            "order_date",
            F.expr("date_sub(current_date(), cast(rand() * 365 as int))")
        )
        .withColumn("year", F.year("order_date"))
        .withColumn("month", F.month("order_date"))
        .withColumn("day", F.dayofmonth("order_date"))
        .select(
            "order_id",
            "customer_id",
            "product",
            "price",
            "quantity",
            "total",
            "order_date",
            "year",
            "month",
            "day",
        )
)

orders_df.write \
    .mode("append") \
    .partitionBy("year", "month", "day") \
    .saveAsTable("orders")

In [4]:
output_path = "Files/orders_parquet"

orders_df.write \
    .mode("append") \
    .partitionBy("year", "month", "day") \
    .parquet(output_path)

StatementMeta(, 1bd44b1a-2e99-454e-bced-69e163e75281, 6, Finished, Available, Finished, False)